In [3]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd
import fiona

#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent
data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'
processed_dir = data_dir / 'processed' / 'analysis/'
vri_dir = data_dir / 'external/bc_vri/Sea_To_Sky'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [4]:
vri_gpkg = Path(vri_dir / 'VEG_COMP_LYR_R1_POLY.gpkg')


vri_gdf = gpd.read_file(vri_gpkg)
vri_gdf.shape, vri_gdf.crs, vri_gdf.geom_type.value_counts()

((102373, 193),
 <Projected CRS: EPSG:3005>
 Name: NAD83 / BC Albers
 Axis Info [cartesian]:
 - E[east]: Easting (metre)
 - N[north]: Northing (metre)
 Area of Use:
 - name: Canada - British Columbia.
 - bounds: (-139.04, 48.25, -114.08, 60.01)
 Coordinate Operation:
 - name: British Columbia Albers
 - method: Albers Equal Area
 Datum: North American Datum 1983
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich,
 MultiPolygon    102373
 Name: count, dtype: int64)

In [5]:
fiona.listlayers(vri_gpkg)

['WHSE_FOREST_VEGETATION.VEG_COMP_LYR_R1_POLY']

In [6]:
vri_gdf.head(3)

,FEATURE_ID,MAP_ID,POLYGON_ID,OPENING_IND,OPENING_SOURCE,OPENING_NUMBER,FEATURE_CLASS_SKEY,INVENTORY_STANDARD_CD,POLYGON_AREA,NON_PRODUCTIVE_DESCRIPTOR_CD,...,FOLIAGE_BIOMASS_PER_HA,BARK_BIOMASS_PER_HA,OBJECTID,SE_ANNO_CAD_DATA,FEATURE_AREA_SQM,FEATURE_LENGTH_M,GEOMETRY.AREA,GEOMETRY.LEN,fme_feature_type,geometry
0,17506696,092J083,35435984,N,4,None,843,V,7.1,None,...,9.052,5.042,15813064,None,70192.1737,1873.0207,0.0,0.0,WHSE_FOREST_VEGETATION.VEG_COMP_LYR_R1_POLY,"MULTIPOLYGON (((1179357.113 656012.114, 117933..."
1,17506731,092J083,34235978,N,4,None,843,V,8.5,None,...,15.696,14.148,14395316,None,84782.5007,2203.3012,0.0,0.0,WHSE_FOREST_VEGETATION.VEG_COMP_LYR_R1_POLY,"MULTIPOLYGON (((1178808.389 656135.066, 117880..."
2,17506730,092J083,32815946,N,4,None,843,V,3.7,None,...,11.122,5.599,15214732,None,36655.3230,1085.4012,0.0,0.0,WHSE_FOREST_VEGETATION.VEG_COMP_LYR_R1_POLY,"MULTIPOLYGON (((1178541.849 656039.413, 117851..."


In [7]:
vri_gdf.columns

Index(['FEATURE_ID', 'MAP_ID', 'POLYGON_ID', 'OPENING_IND', 'OPENING_SOURCE',
       'OPENING_NUMBER', 'FEATURE_CLASS_SKEY', 'INVENTORY_STANDARD_CD',
       'POLYGON_AREA', 'NON_PRODUCTIVE_DESCRIPTOR_CD',
       ...
       'FOLIAGE_BIOMASS_PER_HA', 'BARK_BIOMASS_PER_HA', 'OBJECTID',
       'SE_ANNO_CAD_DATA', 'FEATURE_AREA_SQM', 'FEATURE_LENGTH_M',
       'GEOMETRY.AREA', 'GEOMETRY.LEN', 'fme_feature_type', 'geometry'],
      dtype='object', length=193)

In [8]:
vri_gdf.total_bounds

array([1056974.2691    ,  511729.7981    , 1273725.0861    ,
        716082.72810001])

In [9]:
avcan_gpkg = Path(REPO_ROOT/"data/processed/share/avcan_layers.gpkg")

fiona.listlayers(avcan_gpkg)
layers = fiona.listlayers(avcan_gpkg)
stage_a_gdf = gpd.read_file(avcan_gpkg, layer=layers[1])

In [10]:
stage_a_gdf.head(1)

,Slope Mean Percentage,National Park,Unique Fire ID (gid),Majority Cardinal Direction,Patch ID,Patch Area (ha),Min Elevation (m),Year,Subregion,Mean Elevation (m),figure this out,elev_relie,Max Elevation (m),scenario,FireID,Mean Slope Degree,Region,geometry
0,64.76,None,1990_87,SW,2.0,30.32,1066,1990,Garibaldi,1313.43,228.65,481,1547,direct_post,87,32.93,Sea_To_Sky,"MULTIPOLYGON (((-123.08114 49.88075, -123.0819..."


In [11]:
print(f'Stage A CRS: {stage_a_gdf.crs}')
print(f'VRI CRS: {vri_gdf.crs}\n')

stage_a_3005 = stage_a_gdf.to_crs(vri_gdf.crs)
print('Amend CRS')
print(f'New Stage A CRS: {stage_a_3005.crs}')


Stage A CRS: EPSG:4326
VRI CRS: EPSG:3005

Amend CRS
New Stage A CRS: EPSG:3005


In [12]:

# Recommended: fix invalid geometries before overlay/intersection
for gdf in (stage_a_3005, vri_gdf):
    gdf["geometry"] = gdf["geometry"].buffer(0)  # quick validity fix
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty]

# If you need the cleaned versions to persist:
stage_a_3005 = stage_a_3005[stage_a_3005.geometry.notnull() & ~stage_a_3005.geometry.is_empty].copy()
vri_gdf      = vri_gdf[vri_gdf.geometry.notnull() & ~vri_gdf.geometry.is_empty].copy()

In [13]:
stage_a_3005.columns

Index(['Slope Mean Percentage', 'National Park', 'Unique Fire ID (gid)',
       'Majority Cardinal Direction', 'Patch ID', 'Patch Area (ha)',
       'Min Elevation (m)', 'Year', 'Subregion', 'Mean Elevation (m)',
       'figure this out', 'elev_relie', 'Max Elevation (m)', 'scenario',
       'FireID', 'Mean Slope Degree', 'Region', 'geometry'],
      dtype='object')

In [14]:
# Keep only the Stage A ID fields you need (adjust column names)
stage_cols = ["Unique Fire ID (gid)", "Patch ID", "Year", "Subregion", "geometry"]
stage = stage_a_3005[stage_cols].copy()

# Prefilter VRI first (speeds up overlay)
vri_candidates = vri_gdf[vri_gdf.intersects(stage.unary_union)].copy()

# Intersection overlay
vri_stage_overlap = gpd.overlay(vri_candidates, stage, how="intersection", keep_geom_type=False)

# Overlap area in hectares (EPSG:3005 is meters)
vri_stage_overlap["ov_area_m2"] = vri_stage_overlap.geometry.area
vri_stage_overlap["ov_area_ha"] = vri_stage_overlap["ov_area_m2"] / 10_000

vri_stage_overlap.shape


/var/folders/bs/_y1b6rb96rb21tm4p3r5fpz80000gn/T/ipykernel_12962/2412786175.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  vri_candidates = vri_gdf[vri_gdf.intersects(stage.unary_union)].copy()


(3839, 199)

In [15]:
vri_stage_overlap.head(2)

,FEATURE_ID,MAP_ID,POLYGON_ID,OPENING_IND,OPENING_SOURCE,OPENING_NUMBER,FEATURE_CLASS_SKEY,INVENTORY_STANDARD_CD,POLYGON_AREA,NON_PRODUCTIVE_DESCRIPTOR_CD,...,GEOMETRY.AREA,GEOMETRY.LEN,fme_feature_type,Unique Fire ID (gid),Patch ID,Year,Subregion,geometry,ov_area_m2,ov_area_ha
0,2054556,092N037,16,N,None,None,843,F,149.8,NPBU,...,0.0,0.0,WHSE_FOREST_VEGETATION.VEG_COMP_LYR_R1_POLY,1995_1377,1.0,1995,Homathko,"POLYGON ((1085873.001 702479.408, 1085860.008 ...",135493.064604,13.549306
1,2054556,092N037,16,N,None,None,843,F,149.8,NPBU,...,0.0,0.0,WHSE_FOREST_VEGETATION.VEG_COMP_LYR_R1_POLY,1995_1377,2.0,1995,Homathko,"MULTIPOLYGON (((1085285.012 702968.268, 108530...",38099.326947,3.809933


In [16]:

# ---- 1) columns to carry through as metadata (NOT weighted) ----
VRI_META_FIELDS = [
    "FEATURE_ID", "MAP_ID", "POLYGON_ID", "REFERENCE_DATE"
]

# ---- 2) numeric fields you can area-weight ----
VRI_NUMERIC_FIELDS = [
    "CROWN_CLOSURE",
    "BASAL_AREA",
    "VRI_LIVE_STEMS_PER_HA",
    "SITE_INDEX",
    "PROJ_AGE_1",
    "PROJ_HEIGHT_1",
    "LIVE_VOL_PER_HA_SPP1_125",
    # species percent columns are numeric and can be weighted
    "SPECIES_PCT_1","SPECIES_PCT_2","SPECIES_PCT_3",
    "SPECIES_PCT_4","SPECIES_PCT_5","SPECIES_PCT_6",
]

# ---- 3) categorical / code fields (summarize by overlap area, not mean) ----
VRI_CODE_FIELDS = [
    "BCLCS_LEVEL_1","BCLCS_LEVEL_2","BCLCS_LEVEL_3","BCLCS_LEVEL_4","BCLCS_LEVEL_5",
    "PROJ_AGE_CLASS_CD_1","PROJ_HEIGHT_CLASS_CD_1",
    "SPECIES_CD_1","SPECIES_CD_2","SPECIES_CD_3",
    "SPECIES_CD_4","SPECIES_CD_5","SPECIES_CD_6",
]


In [17]:
# Ensure numeric columns are numeric (bad values become NaN)
for c in VRI_NUMERIC_FIELDS:
    if c in vri_stage_overlap.columns:
        vri_stage_overlap[c] = pd.to_numeric(vri_stage_overlap[c], errors="coerce")


In [18]:
group_cols = ["Unique Fire ID (gid)", "Patch ID", "Year", "Subregion"]

def weighted_means(df, cols):
    w = df["ov_area_m2"].to_numpy()
    out = {}
    wsum = np.nansum(w)
    out["stageA_area_ha"] = df["ov_area_ha"].sum()

    if wsum == 0:
        for c in cols:
            out[c] = np.nan
        return pd.Series(out)

    for c in cols:
        x = df[c].to_numpy(dtype="float64")
        mask = ~np.isnan(x)
        if mask.any():
            out[c] = np.nansum(x[mask] * w[mask]) / np.nansum(w[mask])
        else:
            out[c] = np.nan
    return pd.Series(out)

patch_stats_num = (
    vri_stage_overlap
      .groupby(group_cols, dropna=False)
      .apply(lambda df: weighted_means(df, VRI_NUMERIC_FIELDS))
      .reset_index()
)

patch_stats_num.head()


/var/folders/bs/_y1b6rb96rb21tm4p3r5fpz80000gn/T/ipykernel_12962/2262847444.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: weighted_means(df, VRI_NUMERIC_FIELDS))


,Unique Fire ID (gid),Patch ID,Year,Subregion,stageA_area_ha,CROWN_CLOSURE,BASAL_AREA,VRI_LIVE_STEMS_PER_HA,SITE_INDEX,PROJ_AGE_1,PROJ_HEIGHT_1,LIVE_VOL_PER_HA_SPP1_125,SPECIES_PCT_1,SPECIES_PCT_2,SPECIES_PCT_3,SPECIES_PCT_4,SPECIES_PCT_5,SPECIES_PCT_6
0,1990_108,1.0,1990,Brandywine,1.258129,4.215158,20.481624,951.838955,26.867685,33.048566,16.163355,67.193832,95.331048,28.137347,18.607832,8.607832,NaN,NaN
1,1990_118,1.0,1990,Homathko,0.511072,31.487493,52.823002,265.008116,21.426436,230.000000,38.630563,520.000197,100.000000,NaN,NaN,NaN,NaN,NaN
2,1990_118,2.0,1990,Homathko,4.939471,38.291240,53.958924,319.711748,21.599383,222.261466,37.800542,520.691869,99.560527,10.000000,NaN,NaN,NaN,NaN
3,1990_118,3.0,1990,Homathko,1.815630,40.000000,55.238617,292.000000,21.299999,230.000000,38.400002,540.212000,100.000000,NaN,NaN,NaN,NaN,NaN
4,1990_118,4.0,1990,Homathko,85.377123,24.948697,51.725423,444.967903,20.511700,164.213985,32.207920,336.307370,83.642338,17.785815,10.013739,3.274137,1.0,NaN


In [19]:
def area_weighted_mode(df, col):
    s = (
        df.groupby(col, dropna=False)["ov_area_m2"]
          .sum()
          .sort_values(ascending=False)
    )
    return s.index[0] if len(s) else None

patch_stats_cat = (
    vri_stage_overlap
      .groupby(group_cols, dropna=False)
      .apply(lambda df: pd.Series({
          **{f"{c}_mode": area_weighted_mode(df, c) for c in VRI_CODE_FIELDS if c in df.columns}
      }))
      .reset_index()
)

patch_stats_cat.head()


/var/folders/bs/_y1b6rb96rb21tm4p3r5fpz80000gn/T/ipykernel_12962/3072828765.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: pd.Series({


,Unique Fire ID (gid),Patch ID,Year,Subregion,BCLCS_LEVEL_1_mode,BCLCS_LEVEL_2_mode,BCLCS_LEVEL_3_mode,BCLCS_LEVEL_4_mode,BCLCS_LEVEL_5_mode,PROJ_AGE_CLASS_CD_1_mode,PROJ_HEIGHT_CLASS_CD_1_mode,SPECIES_CD_1_mode,SPECIES_CD_2_mode,SPECIES_CD_3_mode,SPECIES_CD_4_mode,SPECIES_CD_5_mode,SPECIES_CD_6_mode
0,1990_108,1.0,1990,Brandywine,V,N,U,SL,SP,2,2,FD,NaN,NaN,NaN,NaN,NaN
1,1990_118,1.0,1990,Homathko,V,T,U,TC,OP,8,5,FD,NaN,NaN,NaN,NaN,NaN
2,1990_118,2.0,1990,Homathko,V,N,U,SL,SP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1990_118,3.0,1990,Homathko,V,T,U,TC,OP,8,5,FD,NaN,NaN,NaN,NaN,NaN
4,1990_118,4.0,1990,Homathko,V,T,U,TC,SP,8,4,FD,NaN,NaN,NaN,NaN,NaN


In [20]:
patch_stats = patch_stats_num.merge(patch_stats_cat, on=group_cols, how="left")
patch_stats.head()


,Unique Fire ID (gid),Patch ID,Year,Subregion,stageA_area_ha,CROWN_CLOSURE,BASAL_AREA,VRI_LIVE_STEMS_PER_HA,SITE_INDEX,PROJ_AGE_1,...,BCLCS_LEVEL_4_mode,BCLCS_LEVEL_5_mode,PROJ_AGE_CLASS_CD_1_mode,PROJ_HEIGHT_CLASS_CD_1_mode,SPECIES_CD_1_mode,SPECIES_CD_2_mode,SPECIES_CD_3_mode,SPECIES_CD_4_mode,SPECIES_CD_5_mode,SPECIES_CD_6_mode
0,1990_108,1.0,1990,Brandywine,1.258129,4.215158,20.481624,951.838955,26.867685,33.048566,...,SL,SP,2,2,FD,NaN,NaN,NaN,NaN,NaN
1,1990_118,1.0,1990,Homathko,0.511072,31.487493,52.823002,265.008116,21.426436,230.000000,...,TC,OP,8,5,FD,NaN,NaN,NaN,NaN,NaN
2,1990_118,2.0,1990,Homathko,4.939471,38.291240,53.958924,319.711748,21.599383,222.261466,...,SL,SP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1990_118,3.0,1990,Homathko,1.815630,40.000000,55.238617,292.000000,21.299999,230.000000,...,TC,OP,8,5,FD,NaN,NaN,NaN,NaN,NaN
4,1990_118,4.0,1990,Homathko,85.377123,24.948697,51.725423,444.967903,20.511700,164.213985,...,TC,SP,8,4,FD,NaN,NaN,NaN,NaN,NaN


In [21]:
species_pairs = [(f"SPECIES_CD_{i}", f"SPECIES_PCT_{i}") for i in range(1,7)]

def dominant_species(df):
    # weighted mean percent for each species slot
    w = df["ov_area_m2"].to_numpy()
    res = {}
    for cd_col, pct_col in species_pairs:
        if pct_col in df.columns:
            x = pd.to_numeric(df[pct_col], errors="coerce").to_numpy()
            mask = ~np.isnan(x)
            res[pct_col] = (x[mask] * w[mask]).sum() / w[mask].sum() if mask.any() else np.nan

    # choose the slot with highest weighted percent
    best_i = max(range(1,7), key=lambda i: (res.get(f"SPECIES_PCT_{i}", np.nan) if not np.isnan(res.get(f"SPECIES_PCT_{i}", np.nan)) else -1))
    return pd.Series({
        "dom_species_cd": area_weighted_mode(df, f"SPECIES_CD_{best_i}") if f"SPECIES_CD_{best_i}" in df.columns else None,
        "dom_species_pct": res.get(f"SPECIES_PCT_{best_i}", np.nan)
    })

patch_species = (
    vri_stage_overlap
      .groupby(group_cols, dropna=False)
      .apply(dominant_species)
      .reset_index()
)

patch_stats = patch_stats.merge(patch_species, on=group_cols, how="left")
patch_stats.head()


/var/folders/bs/_y1b6rb96rb21tm4p3r5fpz80000gn/T/ipykernel_12962/1110686650.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(dominant_species)


,Unique Fire ID (gid),Patch ID,Year,Subregion,stageA_area_ha,CROWN_CLOSURE,BASAL_AREA,VRI_LIVE_STEMS_PER_HA,SITE_INDEX,PROJ_AGE_1,...,PROJ_AGE_CLASS_CD_1_mode,PROJ_HEIGHT_CLASS_CD_1_mode,SPECIES_CD_1_mode,SPECIES_CD_2_mode,SPECIES_CD_3_mode,SPECIES_CD_4_mode,SPECIES_CD_5_mode,SPECIES_CD_6_mode,dom_species_cd,dom_species_pct
0,1990_108,1.0,1990,Brandywine,1.258129,4.215158,20.481624,951.838955,26.867685,33.048566,...,2,2,FD,NaN,NaN,NaN,NaN,NaN,FD,95.331048
1,1990_118,1.0,1990,Homathko,0.511072,31.487493,52.823002,265.008116,21.426436,230.000000,...,8,5,FD,NaN,NaN,NaN,NaN,NaN,FD,100.000000
2,1990_118,2.0,1990,Homathko,4.939471,38.291240,53.958924,319.711748,21.599383,222.261466,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.560527
3,1990_118,3.0,1990,Homathko,1.815630,40.000000,55.238617,292.000000,21.299999,230.000000,...,8,5,FD,NaN,NaN,NaN,NaN,NaN,FD,100.000000
4,1990_118,4.0,1990,Homathko,85.377123,24.948697,51.725423,444.967903,20.511700,164.213985,...,8,4,FD,NaN,NaN,NaN,NaN,NaN,FD,83.642338


In [22]:
bad = []
for c in VRI_NUMERIC_FIELDS:
    if c in vri_stage_overlap.columns and vri_stage_overlap[c].dtype == "object":
        bad.append(c)
bad


[]

In [23]:
patch_stats.columns

Index(['Unique Fire ID (gid)', 'Patch ID', 'Year', 'Subregion',
       'stageA_area_ha', 'CROWN_CLOSURE', 'BASAL_AREA',
       'VRI_LIVE_STEMS_PER_HA', 'SITE_INDEX', 'PROJ_AGE_1', 'PROJ_HEIGHT_1',
       'LIVE_VOL_PER_HA_SPP1_125', 'SPECIES_PCT_1', 'SPECIES_PCT_2',
       'SPECIES_PCT_3', 'SPECIES_PCT_4', 'SPECIES_PCT_5', 'SPECIES_PCT_6',
       'BCLCS_LEVEL_1_mode', 'BCLCS_LEVEL_2_mode', 'BCLCS_LEVEL_3_mode',
       'BCLCS_LEVEL_4_mode', 'BCLCS_LEVEL_5_mode', 'PROJ_AGE_CLASS_CD_1_mode',
       'PROJ_HEIGHT_CLASS_CD_1_mode', 'SPECIES_CD_1_mode', 'SPECIES_CD_2_mode',
       'SPECIES_CD_3_mode', 'SPECIES_CD_4_mode', 'SPECIES_CD_5_mode',
       'SPECIES_CD_6_mode', 'dom_species_cd', 'dom_species_pct'],
      dtype='object')

In [24]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)          # or a big number like 2000
pd.set_option("display.max_colwidth", None)   # prevents truncation of long strings

patch_stats.head(5)

,Unique Fire ID (gid),Patch ID,Year,Subregion,stageA_area_ha,CROWN_CLOSURE,BASAL_AREA,VRI_LIVE_STEMS_PER_HA,SITE_INDEX,PROJ_AGE_1,PROJ_HEIGHT_1,LIVE_VOL_PER_HA_SPP1_125,SPECIES_PCT_1,SPECIES_PCT_2,SPECIES_PCT_3,SPECIES_PCT_4,SPECIES_PCT_5,SPECIES_PCT_6,BCLCS_LEVEL_1_mode,BCLCS_LEVEL_2_mode,BCLCS_LEVEL_3_mode,BCLCS_LEVEL_4_mode,BCLCS_LEVEL_5_mode,PROJ_AGE_CLASS_CD_1_mode,PROJ_HEIGHT_CLASS_CD_1_mode,SPECIES_CD_1_mode,SPECIES_CD_2_mode,SPECIES_CD_3_mode,SPECIES_CD_4_mode,SPECIES_CD_5_mode,SPECIES_CD_6_mode,dom_species_cd,dom_species_pct
0,1990_108,1.0,1990,Brandywine,1.258129,4.215158,20.481624,951.838955,26.867685,33.048566,16.163355,67.193832,95.331048,28.137347,18.607832,8.607832,NaN,NaN,V,N,U,SL,SP,2,2,FD,NaN,NaN,NaN,NaN,NaN,FD,95.331048
1,1990_118,1.0,1990,Homathko,0.511072,31.487493,52.823002,265.008116,21.426436,230.000000,38.630563,520.000197,100.000000,NaN,NaN,NaN,NaN,NaN,V,T,U,TC,OP,8,5,FD,NaN,NaN,NaN,NaN,NaN,FD,100.000000
2,1990_118,2.0,1990,Homathko,4.939471,38.291240,53.958924,319.711748,21.599383,222.261466,37.800542,520.691869,99.560527,10.000000,NaN,NaN,NaN,NaN,V,N,U,SL,SP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.560527
3,1990_118,3.0,1990,Homathko,1.815630,40.000000,55.238617,292.000000,21.299999,230.000000,38.400002,540.212000,100.000000,NaN,NaN,NaN,NaN,NaN,V,T,U,TC,OP,8,5,FD,NaN,NaN,NaN,NaN,NaN,FD,100.000000
4,1990_118,4.0,1990,Homathko,85.377123,24.948697,51.725423,444.967903,20.511700,164.213985,32.207920,336.307370,83.642338,17.785815,10.013739,3.274137,1.0,NaN,V,T,U,TC,SP,8,4,FD,NaN,NaN,NaN,NaN,NaN,FD,83.642338
